In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np

# Set visual style
sns.set_theme(style="whitegrid")
print("Libraries loaded!")

In [ ]:
# 1. Load income data
income_df = pd.read_csv(r'data\ACSST5Y2024.S1901-Data.csv', skiprows=[1])

# 2.Cleanup
income_cols = ['S1901_C01_012E', 'S1901_C02_012E', 'S1901_C03_012E']
for col in income_cols:
    # Remove commas/plus signs and force to numeric
    income_df[col] = income_df[col].astype(str).str.replace(',', '').str.replace('+', '')
    income_df[col] = pd.to_numeric(income_df[col], errors='coerce')

# 3. Subset and Rename
income_df = income_df[['GEO_ID', 'S1901_C01_012E']].copy()
income_df.columns = ['geo_id', 'median_income']

# 4. Extract Zip Code and final drop of NaNs
income_df['zip_code'] = income_df['geo_id'].str.split('US').str[-1]
income_df = income_df.dropna(subset=['median_income'])

print(f"Loaded income data for {len(income_df)} Zip Codes.")
income_df.head()

In [ ]:
# Load the tree data
trees_df = pd.read_csv('data/2015_Street_Tree_Census_-_Tree_Data_20260315.csv')

# Cleanup

# 1. Filter to only 'Alive' trees (we can't measure the health of a stump)
trees_df = trees_df[trees_df['status'] == 'Alive'].copy()

# 2. Keep only the relevant columns (ADDED 'census tract' here so it isn't lost)
trees_df = trees_df[['tree_id', 'health', 'postcode', 'spc_common', 'borough', 'census tract']]

# 3. Ensure Zip Code (postcode) is a string to match the income data
trees_df['postcode'] = trees_df['postcode'].astype(str)

health_map = {'Good': 3, 'Fair': 2, 'Poor': 1}
trees_df['health_numeric'] = trees_df['health'].map(health_map)

# Filter for alive trees ONLY (handles non-living trees)
if 'status' in trees_df.columns:
    trees_alive = trees_df[trees_df['status'] == 'Alive'].copy()
else:
    trees_alive = trees_df.copy()

trees_alive['postcode'] = trees_alive['postcode'].astype(str)

print(f"Loaded {len(trees_df)} living trees.")
trees_df.head()

In [ ]:
# --- 3. TREE DATA AGGREGATION ---
# We use .agg() and then IMMEDIATELY rename the columns to match your later code
tree_stats = trees_alive.groupby('postcode').agg({
    'tree_id': 'count',
    'health_numeric': 'mean',
    'borough': 'first'  
}).reset_index()

tree_stats.columns = ['zip_code', 'tree_count', 'health_numeric', 'borough']

# Calculate % Good Health safely
health_counts = trees_alive.groupby(['postcode', 'health']).size().unstack(fill_value=0)
if 'Good' not in health_counts.columns: health_counts['Good'] = 0
health_counts['total'] = health_counts.sum(axis=1)
health_counts['percent_good'] = (health_counts['Good'] / health_counts['total']) * 100

# Merge the % Good metric back into tree_stats
tree_stats = tree_stats.merge(health_counts[['percent_good']], left_on='zip_code', right_index=True)

# --- 4. MASTER MERGE ---
final_df = pd.merge(tree_stats, income_df, on='zip_code').dropna(subset=['median_income', 'percent_good'])

print(f"Harmonization Complete: {len(final_df)} Zip Codes successfully linked.")
final_df.head()

In [ ]:
# Z-Score filter for outliers

# Remove Zip Codes with very few trees (e.g., less than 50)
filtered_df = final_df[final_df['tree_count'] > 50].copy()

# Filter out extreme income outliers (more than 3 standard deviations away)
z_scores = np.abs(stats.zscore(filtered_df['median_income']))
clean_df = filtered_df[z_scores < 3]

print(f"Removed {len(final_df) - len(clean_df)} outlier Zip Codes.")

In [ ]:

# Create the scatter plot with a regression line
plt.figure(figsize=(12, 8))

# Use 'health_numeric' 
# Also using clean_df here makes the graph much more readable!
sns.regplot(data=clean_df, x='median_income', y='health_numeric', 
            scatter_kws={'alpha':0.5}, line_kws={'color':'red'})

plt.title('Relationship Between Neighborhood Wealth and Tree Health (NYC)', fontsize=16)
plt.xlabel('Median Household Income ($)', fontsize=12)
plt.ylabel('Average Tree Health Score (1=Poor, 3=Good)', fontsize=12)

plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
print(trees_df.columns.tolist())

In [ ]:
import pandas as pd
import os

# 1. Load the NTA mapping file
nta_path = os.path.join('data', '2020_Census_Tracts_to_2020_NTAs_and_CDTAs_Equivalency_20260414.csv')
nta_df = pd.read_csv(nta_path)

# 2. Prepare the 'Bridge' Keys
# Pad NTA tracts to 6 digits (e.g., 1901 -> '001901')
nta_df['CT_Match_Key'] = nta_df['CT2020'].astype(str).str.zfill(6)

# Pad tree tracts to 6 digits (e.g., 739 -> '000739')
trees_df['CT_Match_Key'] = pd.to_numeric(trees_df['census tract'], errors='coerce').fillna(0).astype(int).astype(str).str.zfill(6)

# 3. Merge the datasets
# This adds NTAName (Neighborhood Name) and BoroName to every tree in the dataset
tree_with_neighborhoods = trees_df.merge(
    nta_df[['CT_Match_Key', 'NTACode', 'NTAName', 'BoroName']], 
    on='CT_Match_Key', 
    how='left'
)

# 4. Verify the result
print("Neighborhood match successful. Preview:")
display(tree_with_neighborhoods[['census tract', 'NTAName', 'health']].head())

In [ ]:

def normalize_tract(val):
    try:
        if pd.isna(val) or str(val).lower() == 'nan':
            return None
        # Remove commas and convert to float first to handle '37,402' and '374.02'
        num = float(str(val).replace(',', ''))
        # Convert to int to drop decimals, then to string
        return str(int(num))
    except:
        return None

# Apply the numeric normalization
trees_df['CT_Normalized'] = trees_df['census tract'].apply(normalize_tract)

# Check which column to use in NTA file
tract_col = 'CT2020' if 'CT2020' in nta_df.columns else 'CT2010'
nta_df['CT_Normalized'] = nta_df[tract_col].apply(normalize_tract)

# --- 2. RE-MERGE NEIGHBORHOODS ---
tree_with_neighborhoods = trees_df.merge(
    nta_df[['CT_Normalized', 'NTACode', 'NTAName', 'BoroName']], 
    on='CT_Normalized', 
    how='inner'
)

# --- 3. MAP INCOME ---
tree_with_neighborhoods['postcode'] = tree_with_neighborhoods['postcode'].astype(str).str.strip()
clean_df['zip_code'] = clean_df['zip_code'].astype(str).str.strip()

trees_with_income = tree_with_neighborhoods.merge(
    clean_df[['zip_code', 'median_income']], 
    left_on='postcode', 
    right_on='zip_code', 
    how='inner'
)

# --- 4. AGGREGATE ---
neighborhood_stats = trees_with_income.groupby(['NTACode', 'NTAName', 'BoroName'], as_index=False).agg(
    avg_health=('health_numeric', 'mean'),
    avg_income=('median_income', 'mean'),
    tree_count=('tree_id', 'count')
)

# --- 5. CALCULATE CORRELATIONS ---
def get_corr(df):
    if len(df) > 3: # Lowered threshold slightly to include more neighborhoods
        return df['avg_income'].corr(df['avg_health'])
    return 0

borough_corrs = neighborhood_stats.groupby('BoroName').apply(
    get_corr, include_groups=False
).reset_index()
borough_corrs.columns = ['BoroName', 'Boro_Correlation']

neighborhood_correlations = neighborhood_stats.merge(borough_corrs, on='BoroName')

print(f"Total Trees Successfully Matched: {len(trees_with_income):,}")
print(f"Total Neighborhoods for Map: {len(neighborhood_correlations)}")
display(neighborhood_correlations.sort_values(by='avg_income', ascending=False).head())

# --- 6. PREP FOR FOLIUM ---

# 1. Create a friendly 'Correlation Description' for map popups
def describe_corr(val):
    if val > 0.3: return "Strong Positive (Wealth = Health)"
    if val > 0.1: return "Weak Positive"
    if val < -0.1: return "Inverse Relationship"
    return "Neutral (No Correlation)"

neighborhood_correlations['relationship_type'] = neighborhood_correlations['Boro_Correlation'].apply(describe_corr)

# 2. Check the spread of our new data
print("Neighborhoods per Borough:")
print(neighborhood_correlations['BoroName'].value_counts())

# 3. Preview the Top 5 'Wealth = Health' Neighborhoods
print("\nTop Neighborhoods with Strongest Income-to-Health Links:")
display(neighborhood_correlations[['NTAName', 'BoroName', 'Boro_Correlation', 'relationship_type']]
        .sort_values(by='Boro_Correlation', ascending=False).head())

In [ ]:
# Calculate the correlation matrix using the cleaned data
# We use 'health_numeric' because that is the name defined in our harmonization block
correlation = clean_df['median_income'].corr(clean_df['health_numeric'])

print(f"The Correlation between Income and Tree Health (Cleaned) is: {correlation:.2f}")

if correlation > 0.3:
    print("Result: There is a moderate to strong positive relationship!")
elif correlation > 0.1:
    print("Result: There is a weak positive relationship.")
elif correlation > -0.1:
    print("Result: There is virtually no linear relationship across the whole city.")
else:
    print("Result: There is a negative relationship.")

# Double check: Also look at the % Good Health correlation
corr_percent = clean_df['median_income'].corr(clean_df['percent_good'])
print(f"The Correlation for % Good Health is: {corr_percent:.2f}")

In [ ]:
# Scatter plot for Income vs. Total Tree Count
plt.figure(figsize=(12, 8))
sns.regplot(data=final_df, x='median_income', y='tree_count', 
            scatter_kws={'alpha':0.5}, line_kws={'color':'green'})

plt.title('Neighborhood Wealth vs. Number of Street Trees', fontsize=16)
plt.xlabel('Median Household Income ($)', fontsize=12)
plt.ylabel('Total Number of Trees', fontsize=12)

plt.show()

In [ ]:
# 1. Prepare health counts safely
# If 'status' is missing, it means we likely already filtered for 'Alive' trees earlier
if 'status' in trees_df.columns:
    health_counts_df = trees_df[trees_df['status'] == 'Alive'].copy()
else:
    health_counts_df = trees_df.copy()

# Ensure postcode is a string to match the census data zip_code
health_counts_df['postcode'] = health_counts_df['postcode'].astype(str)

# 2. Pivot the data to get Good/Fair/Poor counts per Zip
health_pivot = health_counts_df.groupby(['postcode', 'health']).size().unstack(fill_value=0)

# 3. Calculate the percentage
# We check if 'Good' exists in case a zip code has only 'Fair' or 'Poor' trees
if 'Good' in health_pivot.columns:
    health_pivot['total'] = health_pivot.sum(axis=1)
    health_pivot['percent_good'] = (health_pivot['Good'] / health_pivot['total']) * 100
else:
    health_pivot['percent_good'] = 0

# 4. MERGE: Add the new metric to final_df
# We drop the column first if it already exists to avoid '.x' or '.y' suffixes
if 'percent_good' in final_df.columns:
    final_df = final_df.drop(columns=['percent_good'])

final_df = final_df.merge(
    health_pivot[['percent_good']], 
    left_on='zip_code', 
    right_index=True, 
    how='left'
)

# 5. Plotting
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
sns.regplot(data=final_df, x='median_income', y='percent_good', 
            scatter_kws={'alpha':0.4}, line_kws={'color':'green'})

plt.title('Neighborhood Wealth vs. Tree Health (% Good)', fontsize=16)
plt.xlabel('Median Household Income ($)', fontsize=12)
plt.ylabel('% of Trees in "Good" Health', fontsize=12)
plt.show()

In [ ]:
# We haven't found a correlations so far...
# Analyzing the whole city at once mixes concrete jungles (Manhattan) with garden boroughs (Queens). 
# Lets try splitting the scatter plot by Borough.

# --- STEP 1: ADD BOROUGH INFO ---
# Create the lookup from the original trees_df 
zip_to_borough = trees_df.groupby('postcode')['borough'].first().reset_index()
zip_to_borough.columns = ['zip_code', 'borough']

# Merge it into final_df
if 'borough' not in final_df.columns:
    final_df = final_df.merge(zip_to_borough, on='zip_code', how='left')

# --- STEP 2: CREATE THE CLEAN VERSION ---
# We remove Zip Codes with < 20 trees and income outliers

# 1. Filter by tree count
clean_df = final_df[final_df['tree_count'] > 20].copy()

# 2. Filter by Income (removing top/bottom 1% or using Z-score)
z_scores = np.abs(stats.zscore(clean_df['median_income']))
clean_df = clean_df[z_scores < 3]

print(f"Cleaned data ready. Using {len(clean_df)} neighborhoods out of {len(final_df)}.")

# --- STEP 3: BOROUGH CORRELATION ---
print("\n--- Correlation by Borough (Wealth vs. % Good Health) ---")
for borough in clean_df['borough'].dropna().unique():
    subset = clean_df[clean_df['borough'] == borough]
    # We need at least a few data points to calculate correlation
    if len(subset) > 5:
        corr = subset['median_income'].corr(subset['percent_good'])
        print(f"{borough:15} | Correlation: {corr:+.2f} | Zip Codes: {len(subset)}")

# In the Bronx, we have a correlation of +0.46, which is statistically significant
# This suggests that in the Bronx, private investment or local stewardship could be related to socioeconomic status

In [ ]:
# This calculates the correlation for EACH borough separately
for borough in clean_df['borough'].unique():
    subset = clean_df[clean_df['borough'] == borough]
    corr = subset['median_income'].corr(subset['percent_good'])
    print(f"Correlation in {borough}: {corr:.2f}")

In [ ]:
# Create a side-by-side comparison of the Bronx and Manhattan
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Plot 1: The Bronx (The Strong Link)
bronx_data = clean_df[clean_df['borough'] == 'Bronx']
sns.regplot(data=bronx_data, x='median_income', y='percent_good', 
            ax=ax1, color='orange', line_kws={'color':'red'})
ax1.set_title(f'The Bronx: Strong Correlation (+0.46)\nWealth = Health', fontsize=14)

# Plot 2: Manhattan (The Concrete Jungle)
manhattan_data = clean_df[clean_df['borough'] == 'Manhattan']
sns.regplot(data=manhattan_data, x='median_income', y='percent_good', 
            ax=ax2, color='gray', line_kws={'color':'blue'})
ax2.set_title(f'Manhattan: No Correlation (-0.01)\nDensity Levels the Playing Field', fontsize=14)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# ANALYSIS A: SPECIES DIVERSITY (RICHNESS)
# ==========================================================

# 1. Calculate how many unique species exist in each Zip Code
species_diversity = trees_df.groupby('postcode')['spc_common'].nunique().reset_index()
species_diversity.columns = ['zip_code', 'species_diversity']

# 2. Add this diversity score to our master table
final_df = final_df.merge(species_diversity, on='zip_code', how='left')

# 3. Visualization: Correlation between Income and Variety
plt.figure(figsize=(10, 6))
sns.regplot(data=final_df, x='median_income', y='species_diversity', 
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'})

plt.title('Does Neighborhood Wealth Predict Tree Variety?', fontsize=15)
plt.xlabel('Median Household Income ($)')
plt.ylabel('Number of Unique Tree Species')
plt.show()

# 4. Statistical Result
corr = final_df['median_income'].corr(final_df['species_diversity'])
print(f"Overall City Correlation: {corr:.2f}")

In [ ]:
# ==========================================================
# ANALYSIS A.2: THE DIVERSITY WEALTH GAP
# ==========================================================

# 1. Identify the 5 Wealthiest and 5 Poorest Zip Codes
poorest_5 = final_df.nsmallest(5, 'median_income').copy()
wealthiest_5 = final_df.nlargest(5, 'median_income').copy()

# 2. Add a label so we can tell them apart in the chart
poorest_5['Group'] = 'Bottom 5 (Poorest)'
wealthiest_5['Group'] = 'Top 5 (Wealthiest)'

# 3. Combine them into one temporary table for plotting
comparison_df = pd.concat([poorest_5, wealthiest_5])

# 4. Visualization: Bar Chart
plt.figure(figsize=(12, 6))
sns.barplot(
    data=comparison_df.sort_values('median_income'), 
    x='zip_code', 
    y='species_diversity', 
    hue='Group',
    palette={'Bottom 5 (Poorest)': 'salmon', 'Top 5 (Wealthiest)': 'seagreen'}
)

plt.title('The Species Diversity Gap: Richest vs. Poorest Neighborhoods', fontsize=16)
plt.xlabel('Zip Code', fontsize=12)
plt.ylabel('Number of Unique Tree Species', fontsize=12)
plt.legend(title='Neighborhood Group')
plt.xticks(rotation=45)
plt.show()

# 5. Calculation: The "Variety Gap" Percentage
avg_poor = poorest_5['species_diversity'].mean()
avg_rich = wealthiest_5['species_diversity'].mean()
gap = ((avg_rich - avg_poor) / avg_poor) * 100

print(f"Average species in poorest 5 Zips: {avg_poor:.1f}")
print(f"Average species in wealthiest 5 Zips: {avg_rich:.1f}")
print(f"Outcome: Wealthy neighborhoods have {gap:.1f}% more tree variety than poor ones.")

In [ ]:
# ==========================================================
# ANALYSIS B: DIVERSITY TRENDS BY BOROUGH
# ==========================================================

# 1. Filter out "Extreme" Zip Codes (those with way too many or too few trees)
# This prevents a massive park or a tiny industrial lot from skewing the results
mean_trees = final_df['tree_count'].mean()
std_trees = final_df['tree_count'].std()

close_avg_df = final_df[
    (final_df['tree_count'] >= mean_trees - std_trees) & 
    (final_df['tree_count'] <= mean_trees + std_trees)
].copy()

# 2. Calculate Correlation for each Borough individually
print("Correlation by Borough (Income vs. Species Variety):")
for borough in close_avg_df['borough'].dropna().unique():
    subset = close_avg_df[close_avg_df['borough'] == borough]
    if len(subset) > 5:
        borough_corr = subset['median_income'].corr(subset['species_diversity'])
        print(f" - {borough:15} | Correlation: {borough_corr:+.2f}")

In [ ]:
# ==========================================================
# ANALYSIS C: MOST RESILIENT SPECIES BY BOROUGH
# ==========================================================

# 1. Calculate health stats for every species in every borough
species_borough_health = trees_df.groupby(['borough', 'spc_common']).agg(
    tree_count=('tree_id', 'count'),
    avg_health_score=('health_numeric', 'mean')
).reset_index()

# 2. Filter: Only look at species with at least 200 trees (for statistical reliability)
species_borough_health = species_borough_health[species_borough_health['tree_count'] >= 200]

# 3. Visualization: Focus on the Bronx (per your project goal)
target_borough = 'Bronx'
borough_top_15 = species_borough_health[
    species_borough_health['borough'] == target_borough
].sort_values(by='avg_health_score', ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(data=borough_top_15, x='avg_health_score', y='spc_common', palette='viridis')
plt.title(f'Top 15 Healthiest Tree Species in the {target_borough}', fontsize=15)
plt.xlabel('Average Health Score (3=Good, 1=Poor)')
plt.ylabel('Tree Species Name')
plt.xlim(1, 3) # Set scale to match our 1-3 health scores
plt.show()

In [ ]:
# 1. Merge individual tree records with income data
# This gives every single tree a 'median_income' value based on its Zip Code
trees_with_income = trees_df.merge(income_df, left_on='postcode', right_on='zip_code')

# 2. Create a list to store our results
results = []

# 3. Loop through every species in every borough
# Group by species and borough to calculate the correlation for that specific pair
for (species, borough), group in trees_with_income.groupby(['spc_common', 'borough']):
    
    # We only calculate if we have enough trees (50 or more)to make the result statistically valid
    if len(group) > 50:
        # Calculate the correlation between this specific group's health and their neighborhood's income
        correlation = group['health_numeric'].corr(group['median_income'])
        
        results.append({
            'species': species,
            'borough': borough,
            'correlation': correlation
        })
corr_df = pd.DataFrame(results)

print(f"Correlation Engine Complete. Calculated relationships for {len(corr_df)} species-borough pairs.")

In [ ]:
# ==========================================================
# ANALYSIS D: THE "CLEAN" MASTER HEATMAP
# ==========================================================

# 1. Identify the Top 20 most common species city-wide to reduce clutter
top_20_species = trees_df['spc_common'].value_counts().head(20).index

# 2. Re-filter our correlation data to only include these top species
filtered_corr_df = corr_df[corr_df['species'].isin(top_20_species)]

# 3. Create the Pivot Table
heatmap_df = filtered_corr_df.pivot(index='species', columns='borough', values='correlation')

# 4. Visualization
plt.figure(figsize=(10, 10)) # Smaller, more square size is easier to read

sns.heatmap(
    heatmap_df, 
    cmap='coolwarm',     # Red = Positive correlation, Blue = Negative
    center=0, 
    annot=True,          # Put the actual numbers in the boxes
    fmt=".2f",           # Limit numbers to 2 decimal places
    linewidths=1, 
    cbar_kws={'label': 'Correlation Coefficient'}
)

plt.title('How Neighborhood Wealth Impacts NYC\'s Top 20 Tree Species', fontsize=16, pad=20)
plt.xlabel('Borough', fontsize=12)
plt.ylabel('Common Tree Species', fontsize=12)
plt.tight_layout() # Prevents label cutoff
plt.show()
# Closer to red means more correlation, so with these trees in this borough we see a strong correlation.
# Species that have a strong correlation are considered vulernable, meaning we should avoid planting these in low income areas. In low income areas, prefer trees in the white or blue color range.

